In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 and restart.")

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
def train_model(model, train_loader, test_loader, epochs, lr=0.001, model_name="model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "test_accuracy": [], "epoch_time": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        start = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        accuracy = evaluate_model(model, test_loader)
        epoch_time = time.time() - start

        history["train_loss"].append(avg_loss)
        history["test_accuracy"].append(accuracy)
        history["epoch_time"].append(epoch_time)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} — loss: {avg_loss:.3f} — test accuracy: {accuracy:.2f}% — {epoch_time:.1f}s")

    return history


def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
print("=" * 50)
print("PART A: Training CNN from scratch")
print("=" * 50)

cnn_model = SimpleCNN(num_classes=10)
cnn_history = train_model(cnn_model, train_loader, test_loader, epochs=10, lr=0.001, model_name="SimpleCNN")

cnn_final_accuracy = cnn_history["test_accuracy"][-1]
cnn_total_time = sum(cnn_history["epoch_time"])
print(f"\nSimpleCNN final test accuracy: {cnn_final_accuracy:.2f}%")
print(f"SimpleCNN total training time: {cnn_total_time:.0f}s")

In [ ]:
print("\n" + "=" * 50)
print("PART B: Fine-tuning pretrained ResNet18")
print("=" * 50)

resnet_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

for name, param in resnet_model.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 10)

resnet_history = train_model(resnet_model, train_loader, test_loader, epochs=5, lr=0.0005, model_name="ResNet18")

resnet_final_accuracy = resnet_history["test_accuracy"][-1]
resnet_total_time = sum(resnet_history["epoch_time"])
print(f"\nResNet18 (fine-tuned) final test accuracy: {resnet_final_accuracy:.2f}%")
print(f"ResNet18 total training time: {resnet_total_time:.0f}s")

In [ ]:
print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"SimpleCNN (from scratch, 10 epochs)   : {cnn_final_accuracy:.2f}% accuracy, {cnn_total_time:.0f}s total")
print(f"ResNet18 (fine-tuned, 5 epochs)       : {resnet_final_accuracy:.2f}% accuracy, {resnet_total_time:.0f}s total")
print(f"Accuracy improvement                  : {resnet_final_accuracy - cnn_final_accuracy:+.2f} percentage points")
print(f"Wall-clock time difference            : ResNet18 took {resnet_total_time/cnn_total_time:.1f}x {'LONGER' if resnet_total_time > cnn_total_time else 'less'} time")
print("=" * 50)
print("\nIMPORTANT NUANCE: ResNet18 needed fewer EPOCHS, but each epoch is far")
print("more computationally expensive (deeper network) than the small CNN's")
print("epoch. So transfer learning here traded LESS data/epochs needed for")
print("MORE wall-clock compute per epoch — it did not reduce total training time.")

In [ ]:
torch.save(cnn_model.state_dict(), "simple_cnn.pth")
torch.save(resnet_model.state_dict(), "resnet18_finetuned.pth")
print("\nModels saved: simple_cnn.pth, resnet18_finetuned.pth")
print("On Kaggle: find them in the Output tab on the right sidebar after the notebook finishes running.")